In [ ]:
!pip install transformers torch pandas scikit-learn openpyxl

In [1]:
import torch
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

# Movie Review Sentiment Dataset
sentences = [
    # Positive
    "This movie was absolutely fantastic and I loved every moment of it",
    "An incredible performance by the entire cast brilliant storytelling",
    "One of the best films I have ever seen truly a masterpiece",
    "The plot was gripping and the visuals were stunning throughout",
    "A heartwarming story that left me smiling for days",
    "Exceptional direction and a wonderfully crafted screenplay",
    "The actors delivered powerful performances in every scene",
    "A beautiful and moving film that touched my heart deeply",
    "Loved the chemistry between the characters absolutely delightful",
    "A must watch film with breathtaking cinematography and great music",
    "The film exceeded all my expectations truly outstanding work",
    "Brilliantly written with a perfect blend of drama and humor",
    "I was on the edge of my seat the entire time thrilling",
    "A touching and emotional journey that resonated deeply with me",
    "The best movie of the year without any doubt whatsoever",
    "Superb acting and a compelling story that keeps you hooked",
    "An unforgettable cinematic experience full of emotion and depth",
    "The screenplay was witty intelligent and thoroughly entertaining",
    "Every scene was crafted with such care and attention to detail",
    "A wonderful film that the whole family can enjoy together",
    "The soundtrack perfectly complemented the emotional storyline",
    "Stunning visuals combined with a deeply moving narrative",
    "The director created something truly special and unique here",
    "A flawless performance that deserves every award it receives",
    "This film will stay with me for a very long time incredible",
    # Negative
    "This movie was a complete waste of my time and money",
    "Terrible acting and a plot that made absolutely no sense at all",
    "One of the worst films I have ever had the misfortune of watching",
    "Boring predictable and utterly disappointing from start to finish",
    "The story was all over the place and hard to follow",
    "Poor direction and weak characters ruined the entire experience",
    "I fell asleep halfway through it was that dull and uninteresting",
    "A confusing mess with no clear storyline or character development",
    "The dialogue was cringe worthy and the acting was very wooden",
    "Completely overrated and did not live up to any of the hype",
    "The special effects were laughably bad and cheaply done",
    "A dreadful script that insults the intelligence of the audience",
    "Nothing made sense and the ending was deeply unsatisfying",
    "The worst screenplay I have encountered in recent memory",
    "Flat characters zero chemistry and a storyline going nowhere",
    "An absolute disaster of a film that should never have been made",
    "I cannot believe how much time was wasted on this pointless film",
    "Poorly edited with scenes that had no connection to each other",
    "The movie had no redeeming qualities whatsoever avoid at all costs",
    "A tedious and joyless experience that drags on for far too long",
    "The plot holes were enormous and the logic was completely absent",
    "Uninspired and derivative with nothing new or interesting to offer",
    "The casting was all wrong and the performances were unconvincing",
    "A monumental failure on every single level of filmmaking",
    "I regret watching this film it was an utter disappointment",
]

labels = ["positive"] * 25 + ["negative"] * 25

df = pd.DataFrame({"text": sentences, "label": labels})
print(f"Dataset size: {len(df)}")
print(df["label"].value_counts())

Dataset size: 50
label
positive    25
negative    25
Name: count, dtype: int64


In [2]:
# Label encode
encoder = LabelEncoder()
df["label_id"] = encoder.fit_transform(df["label"])
print("Classes:", encoder.classes_)  # ['negative'=0, 'positive'=1]

# Train/val split
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["text"].tolist(),
    df["label_id"].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df["label_id"]
)

# Load DistilBERT tokenizer (lighter than BERT, faster on Colab)
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
val_encodings   = tokenizer(val_texts,   truncation=True, padding=True, max_length=128)

print(f"Train size: {len(train_texts)} | Val size: {len(val_texts)}")

Classes: ['negative' 'positive']


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Train size: 40 | Val size: 10


In [3]:
class SentimentDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = SentimentDataset(train_encodings, train_labels)
val_dataset   = SentimentDataset(val_encodings,   val_labels)
print("Datasets ready!")

Datasets ready!


In [5]:
# Load pretrained DistilBERT for binary classification
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",        # ← fixed (was evaluation_strategy)
    save_strategy="no",
    logging_dir="./logs",
    logging_steps=10,
    load_best_model_at_end=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

trainer.train()
print("Training complete!")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but n

Epoch,Training Loss,Validation Loss
1,No log,0.603011
2,0.597382,0.427936
3,0.597382,0.341784
4,0.269549,0.321079


Training complete!


In [6]:
# Evaluate
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

def predict_sentiment(text):
    inputs = tokenizer(text, return_tensors="pt",
                       truncation=True, padding=True, max_length=128)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
    predicted_class = torch.argmax(outputs.logits, dim=1).item()
    confidence = torch.softmax(outputs.logits, dim=1).max().item()
    label = encoder.inverse_transform([predicted_class])[0]
    return label, round(confidence * 100, 2)

# Test on new sentences
test_sentences = [
    "This was an absolutely wonderful experience I loved it",
    "Terrible movie I hated every single minute of it",
    "The acting was superb and the story was deeply moving",
    "Boring and pointless I want my money back",
    "A masterpiece of modern cinema truly breathtaking",
]

print("── Sentiment Prediction Results ──\n")
for sentence in test_sentences:
    label, confidence = predict_sentiment(sentence)
    emoji = "😊" if label == "positive" else "😞"
    print(f"{emoji} [{label.upper():8s}] ({confidence}%) → \"{sentence}\"")

── Sentiment Prediction Results ──

😊 [POSITIVE] (86.23%) → "This was an absolutely wonderful experience I loved it"
😞 [NEGATIVE] (78.54%) → "Terrible movie I hated every single minute of it"
😊 [POSITIVE] (86.55%) → "The acting was superb and the story was deeply moving"
😞 [NEGATIVE] (79.6%) → "Boring and pointless I want my money back"
😊 [POSITIVE] (86.77%) → "A masterpiece of modern cinema truly breathtaking"
